# Vector Field Navigation Literature Review
## Comprehensive Analysis for Paper_Draft_6 Introduction

This notebook:
1. Analyzes 57 PDFs from Papers/ folder
2. Filters out irrelevant papers (especially artificial potential fields)
3. Extracts claims from your current intro
4. Matches citations to claims with full traceability
5. Generates expanded paragraph 3 with comprehensive vector field literature
6. Catalogs ALL vector field navigation papers (Kitts + non-Kitts)

**Key Feature:** Every citation includes exact filename and absolute file path for easy sourcing

## Cell 1: Setup & Imports

In [1]:
# Import required libraries
import os
import glob
import json
import re
from typing import List, Dict, Tuple
import time
from pathlib import Path

# LangChain and OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Helper function to extract JSON from LLM responses
def extract_json(text):
    """Extract JSON (object or array) from potentially messy LLM response"""
    # Try direct parse first
    try:
        return json.loads(text)
    except:
        pass
    
    # Look for JSON in code blocks
    # Try ```json first
    json_match = re.search(r'```json\s*(\{[\s\S]*?\}|\[[\s\S]*?\])\s*```', text)
    if json_match:
        try:
            return json.loads(json_match.group(1))
        except:
            pass
    
    # Try plain ```
    code_match = re.search(r'```\s*(\{[\s\S]*?\}|\[[\s\S]*?\])\s*```', text)
    if code_match:
        try:
            return json.loads(code_match.group(1))
        except:
            pass
    
    # Try to extract first complete JSON object or array
    # Find first { or [
    for i, char in enumerate(text):
        if char == '{':
            # Try to find matching }
            depth = 0
            for j in range(i, len(text)):
                if text[j] == '{':
                    depth += 1
                elif text[j] == '}':
                    depth -= 1
                    if depth == 0:
                        try:
                            return json.loads(text[i:j+1])
                        except:
                            break
        elif char == '[':
            # Try to find matching ]
            depth = 0
            for j in range(i, len(text)):
                if text[j] == '[':
                    depth += 1
                elif text[j] == ']':
                    depth -= 1
                    if depth == 0:
                        try:
                            return json.loads(text[i:j+1])
                        except:
                            break
    
    raise ValueError(f"Could not extract valid JSON from response. Response was: {text[:500]}...")

print("✓ Imports successful")

/Users/christopherwaight/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✓ Imports successful


## Cell 2: Configuration

In [2]:
# Set OpenAI API Key
import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API Key: ")

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0.1)

# Paths
PAPERS_FOLDER = "./Papers"
PAPER_DRAFT_6 = "../Vector Field Paper/Paper_Draft_6.tex"

# Configuration
CONFIG = {
    "relevance_threshold": 4,  # 0-10 scale, keep papers >= 4
    "min_cites_per_claim": 1,
    "max_cites_per_claim": 3,
    "relevance_score_threshold": 0.50,  # 0.00-1.00 scale
}

print("✓ Configuration loaded")
print(f"  Papers folder: {PAPERS_FOLDER}")
print(f"  Paper Draft 6: {PAPER_DRAFT_6}")
print(f"  Relevance threshold: {CONFIG['relevance_threshold']}/10")

✓ Configuration loaded
  Papers folder: ./Papers
  Paper Draft 6: ../Vector Field Paper/Paper_Draft_6.tex
  Relevance threshold: 4/10


## Cell 3: Smart Pre-Filtering (Token Optimization)
Load all 57 PDFs, categorize them, and filter out irrelevant papers (especially artificial potential fields)

In [3]:
print("=" * 80)
print("PHASE 1: SMART PRE-FILTERING")
print("=" * 80)

# Load all PDFs
pdf_paths = glob.glob(f"{PAPERS_FOLDER}/*.pdf")
print(f"\nFound {len(pdf_paths)} PDF files\n")

# Pre-filtering prompt
filter_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are categorizing papers for a vector field navigation research project.
    
    CRITICAL DISTINCTION:
    - PHYSICAL vector fields = actual environmental fields (ocean currents, wind, electromagnetic)
    - ARTIFICIAL vector fields = synthetic potential fields for path planning/obstacle avoidance
    
    We want papers about navigating PHYSICAL vector fields, not artificial potential fields."""),
    ("human", """Categorize this paper and score its relevance:
    
    Paper: {filename}
    Content: {content}
    
    Return ONLY a JSON object with this exact format:
    {{
        "category": "<one of: physical_vector_field | artificial_potential_field | scalar_field | formation_control | multirobot_apps | not_relevant>",
        "relevance_score": <integer 0-10>,
        "reason": "<brief explanation>",
        "is_kitts": <true or false>
    }}
    
    Relevance score guide:
    - 9-10: Directly about physical vector field navigation
    - 7-8: Scalar field methods (for comparison) or formation control discussing fields
    - 5-6: Multirobot applications or general navigation
    - 3-4: Tangentially related
    - 0-2: Not relevant
    
    REJECT (score 0-2): Artificial potential fields, pure path planning, unrelated topics
    """)
])

chain = filter_prompt | llm

# Process each PDF
all_papers = []
kept_papers = []
rejected_papers = []

for i, pdf_path in enumerate(pdf_paths, 1):
    filename = os.path.basename(pdf_path)
    print(f"[{i}/{len(pdf_paths)}] Processing: {filename[:60]}...")
    
    try:
        # Load first 2 pages
        loader = PyPDFLoader(pdf_path)
        pages = loader.load()
        content = ""
        for page in pages[:2]:  # First 2 pages only
            content += page.page_content[:2000]  # Limit per page
        
        # Get categorization
        response = chain.invoke({"filename": filename, "content": content})
        
        # Parse JSON response using robust extract_json function
        result = extract_json(response.content)
        
        paper_info = {
            "filename": filename,
            "path": pdf_path,
            "category": result["category"],
            "relevance_score": result["relevance_score"],
            "reason": result["reason"],
            "is_kitts": result["is_kitts"],
            "content": content
        }
        
        all_papers.append(paper_info)
        
        # Filter based on threshold
        if result["relevance_score"] >= CONFIG["relevance_threshold"]:
            kept_papers.append(paper_info)
            print(f"  ✓ KEPT - Score: {result['relevance_score']}/10 - {result['category']}")
        else:
            rejected_papers.append(paper_info)
            print(f"  ✗ REJECTED - Score: {result['relevance_score']}/10 - {result['reason'][:50]}")
        
        time.sleep(0.3)  # Rate limiting
        
    except Exception as e:
        print(f"  ! ERROR: {e}")
        # Log which paper failed and continue
        continue

print("\n" + "=" * 80)
print("PRE-FILTERING RESULTS")
print("=" * 80)
print(f"Total papers processed: {len(all_papers)}")
print(f"Papers KEPT: {len(kept_papers)}")
print(f"Papers REJECTED: {len(rejected_papers)}")
print(f"Papers FAILED: {len(pdf_paths) - len(all_papers)}")

# Summary by category (kept papers)
print("\nKEPT PAPERS BY CATEGORY:")
categories = {}
for p in kept_papers:
    cat = p["category"]
    if cat not in categories:
        categories[cat] = []
    categories[cat].append(p)

for cat, papers in categories.items():
    print(f"\n{cat.upper().replace('_', ' ')} ({len(papers)} papers):")
    for p in papers:
        kitts_marker = "[KITTS]" if p["is_kitts"] else ""
        print(f"  - {p['filename'][:70]} {kitts_marker}")

# Show rejected papers
print("\n" + "=" * 80)
print("REJECTED PAPERS (saved tokens):")
print("=" * 80)
for p in rejected_papers:
    print(f"  [{p['relevance_score']}/10] {p['filename'][:60]}")
    print(f"      Reason: {p['reason']}")

PHASE 1: SMART PRE-FILTERING

Found 55 PDF files

[1/55] Processing: [2] Citation Trakas P S Tantoulas A Bechlioulis CP Formation...
  ✓ KEPT - Score: 5/10 - formation_control
[2/55] Processing: [9] Distributed Multi-Robot Active-Sensing of a Diffusive So...
  ✓ KEPT - Score: 9/10 - physical_vector_field
[3/55] Processing: [30] Multiple UAV Adaptive Navigation for Three-Dimensional ...
  ✓ KEPT - Score: 7/10 - scalar_field
[4/55] Processing: [13] EntrapmentEscorting and Patrolling Missions - Missions....
  ✓ KEPT - Score: 5/10 - formation_control


Ignoring wrong pointing object 5 0 (offset 0)
Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)


[5/55] Processing: [52] Structured Isosurface Mapping of 3D Scalar Fields - Iso...
  ✓ KEPT - Score: 7/10 - scalar_field
[6/55] Processing: [15] Finite-Time Estimation and Control for Multi-Aircraft S...
  ✓ KEPT - Score: 9/10 - physical_vector_field
[7/55] Processing: [39] Precise Landing of Autonomous Aerial Vehicles Using Vec...
  ✗ REJECTED - Score: 1/10 - The paper discusses the use of artificial velocity
[8/55] Processing: [25] Initial Study of Multirobot Adaptive Navigation for - S...
  ✓ KEPT - Score: 9/10 - physical_vector_field
[9/55] Processing: [50] Spiral Search Pattern for Scalable Assemblages of - Sea...
  ✓ KEPT - Score: 7/10 - scalar_field
[10/55] Processing: [5] Connected and automated vehicles CA Vs and robot swarms ...
  ✗ REJECTED - Score: 1/10 - The paper focuses on small-scale testbeds for conn
[11/55] Processing: [4] Co-Evolution of Multi-Robot Controllers and Task Cues fo...
  ✓ KEPT - Score: 5/10 - multirobot_apps


Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 29 0 (offset 0)
Ignoring wrong pointing object 31 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)
Ignoring wrong pointing object 40 0 (offset 0)
Ignoring wrong pointing object 47 0 (offset 0)


[12/55] Processing: [35] Obstacle Avoidance Policies for Cluster Space Control o...
  ✗ REJECTED - Score: 1/10 - The paper focuses on obstacle avoidance and contro
[13/55] Processing: [7] Cooperative Control of Mobile Sensor Networks Adaptive G...
  ✗ REJECTED - Score: 1/10 - The paper focuses on cooperative control using art
[14/55] Processing: [54] Vector Field Path Following for Miniature Air Vehicles ...
  ✗ REJECTED - Score: 0/10 - The paper discusses the use of vector fields for p
[15/55] Processing: [3] Cluster Space Speciﬁcation and Control of Mobile Multiro...
  ✓ KEPT - Score: 5/10 - formation_control


...

[41/55] Processing: [53] Unifying Control Architecture for Reactive Particle Swa...
  ✗ REJECTED - Score: 3/10 - The paper focuses on swarm robotics and control ar
[42/55] Processing: [26] Circular Formation Control - 2018.pdf...
  ✓ KEPT - Score: 5/10 - formation_control
[43/55] Processing: [47] SICE Journal of Control Measurement and System Integrat...
  ✓ KEPT - Score: 5/10 - formation_control
[44/55] Processing: [11] Dynamic Elliptical Shaping Control for Swarm Robots - K...
  ✗ REJECTED - Score: 1/10 - The paper discusses swarm robot control using arti
[45/55] Processing: [29] Motion Planning and Collision Avoidance using Navigatio...
  ✗ REJECTED - Score: 1/10 - The paper discusses synthetic vector fields for mo
[46/55] Processing: [49] Sparsity Structure and Optimality of Multi-Robot Covera...
  ✓ KEPT - Score: 5/10 - formation_control
[47/55] Processing: [48] Singularity-Free Guiding Vector Field for Robot Navigat...
  ✗ REJECTED - Score: 1/10 - The paper discusses a guiding ve

Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)


[48/55] Processing: [40] Proceedings of the ASME 2019 International Design Engin...
  ✓ KEPT - Score: 9/10 - physical_vector_field
[49/55] Processing: [12] Dynamic Plume T racking by Cooperative Robots Jun-Wei W...
  ✓ KEPT - Score: 9/10 - physical_vector_field
[50/55] Processing: [24] ICA T An Indoor Connected and Autonomous T estbed for -...
  ✗ REJECTED - Score: 1/10 - The paper focuses on indoor autonomous driving tes
[51/55] Processing: [41] Quaternions and Dual Quaternions Singularity-Free Multi...
  ✓ KEPT - Score: 5/10 - formation_control


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)


[52/55] Processing: [36] Obstacle Avoidance Using Complex Vector Fields - Avoida...
  ✗ REJECTED - Score: 1/10 - The paper focuses on obstacle avoidance using arti
[53/55] Processing: [10] Distributed Optimal Formation for Uncertain Euler-Lagra...
  ✓ KEPT - Score: 5/10 - formation_control
[54/55] Processing: [23] Himangshu Kalita Ravi Teja Nallapu Andrew Warren and - ...
  ✓ KEPT - Score: 5/10 - multirobot_apps
[55/55] Processing: [31] Multi-Robot Object SLAM Using Distributed Variational I...
  ✓ KEPT - Score: 5/10 - multirobot_apps

PRE-FILTERING RESULTS
Total papers processed: 55
Papers KEPT: 37
Papers REJECTED: 18
Papers FAILED: 0

KEPT PAPERS BY CATEGORY:

FORMATION CONTROL (9 papers):
  - [2] Citation Trakas P S Tantoulas A Bechlioulis CP Formation Control o 
  - [13] EntrapmentEscorting and Patrolling Missions - Missions.pdf [KITTS]
  - [3] Cluster Space Speciﬁcation and Control of Mobile Multirobot System [KITTS]
  - [57] IDETC_Paper_Final_Cwaight (2).pdf [KITTS]
  - [26] Circ

## Cell 4: Load Current Intro from Paper_Draft_6
Parse the introduction to understand what needs citations

In [4]:
print("=" * 80)
print("PHASE 2: LOAD CURRENT INTRODUCTION")
print("=" * 80)

# Read Paper_Draft_6.tex
with open(PAPER_DRAFT_6, 'r', encoding='utf-8') as f:
    draft_content = f.readlines()

# Extract introduction section (lines 39-52 based on earlier read)
intro_lines = draft_content[38:52]  # 0-indexed
intro_text = ''.join(intro_lines)

# Split into paragraphs
paragraphs = [p.strip() for p in intro_text.split('\n\n') if p.strip() and not p.strip().startswith('%')]

print(f"\nExtracted {len(paragraphs)} paragraphs from introduction\n")

for i, para in enumerate(paragraphs, 1):
    print(f"PARAGRAPH {i}:")
    print("-" * 80)
    print(para[:300] + ("..." if len(para) > 300 else ""))
    print("\n")

# Store for later use
current_intro = {
    "full_text": intro_text,
    "paragraphs": paragraphs
}

print("✓ Introduction loaded successfully")

PHASE 2: LOAD CURRENT INTRODUCTION

Extracted 7 paragraphs from introduction

PARAGRAPH 1:
--------------------------------------------------------------------------------
\section{Introduction}
\IEEEPARstart{M}{ultirobot} systems enable exploration of environments too hazardous or inaccessible for human presence, from disaster zones and chemical spills to deep ocean trenches and atmospheric phenomena [1-3]. These methods have proven effective in applications ranging ...


PARAGRAPH 2:
--------------------------------------------------------------------------------
In scalar fields, every point in space is associated with a single value, such as temperature or elevation. Navigation techniques through scalar fields are well studied, with successful demonstrations across diverse platforms including aerial drones, land rovers, and autonomous underwater vehicles (...


PARAGRAPH 3:
--------------------------------------------------------------------------------
Vector fields, prevalent in 

## Cell 5: Extract Claims Needing Citations
Identify factual statements in each paragraph that need citations

In [5]:
print("=" * 80)
print("PHASE 3: IDENTIFY CITATION POINTS")
print("=" * 80)

# Simple approach: Split paragraphs into sentences
# Each sentence that makes a factual claim needs citations

import re

def split_into_sentences(text):
    """Split text into sentences"""
    # Simple sentence splitter
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

all_claims = []

for para_num, paragraph in enumerate(current_intro["paragraphs"][:3], 1):
    print(f"\nProcessing PARAGRAPH {para_num}...")
    
    # Split into sentences
    sentences = split_into_sentences(paragraph)
    
    print(f"  Found {len(sentences)} sentences")
    
    # Manually categorize by paragraph
    if para_num == 1:
        topic = "multirobot_apps"
    elif para_num == 2:
        topic = "scalar_fields"
    elif para_num == 3:
        topic = "vector_fields"
    else:
        topic = "other"
    
    # Add each sentence as a potential claim
    # Filter out obvious non-citeable sentences
    for sentence in sentences:
        # Skip very short sentences
        if len(sentence) < 20:
            continue
        
        # Skip sentences about "this paper" or "our approach"
        if any(phrase in sentence.lower() for phrase in [
            "this paper", "our approach", "our contribution", "we present",
            "we develop", "we show", "we validate", "section ii", "section iii",
            "the remainder", "we introduce"
        ]):
            continue
        
        # Skip the paper organization sentence
        if "remainder of this paper" in sentence.lower():
            continue
        
        # This is a citeable claim
        all_claims.append({
            "claim": sentence,
            "paragraph": para_num,
            "topic": topic
        })

# Display all claims organized by paragraph
print("\n" + "=" * 80)
print("IDENTIFIED CITATION POINTS BY PARAGRAPH")
print("=" * 80)

for para_num in [1, 2, 3]:
    para_claims = [c for c in all_claims if c["paragraph"] == para_num]
    print(f"\nPARAGRAPH {para_num} ({len(para_claims)} citation points):")
    print("-" * 80)
    for i, claim in enumerate(para_claims, 1):
        print(f"\n[{para_num}.{i}] {claim['claim'][:100]}...")

print(f"\n\n✓ Total citation points identified: {len(all_claims)}")
print("\nNote: These are sentences that need citations. We'll match papers to them in the next step.")

PHASE 3: IDENTIFY CITATION POINTS

Processing PARAGRAPH 1...
  Found 3 sentences

Processing PARAGRAPH 2...
  Found 3 sentences

Processing PARAGRAPH 3...
  Found 6 sentences

IDENTIFIED CITATION POINTS BY PARAGRAPH

PARAGRAPH 1 (3 citation points):
--------------------------------------------------------------------------------

[1.1] \section{Introduction}
\IEEEPARstart{M}{ultirobot} systems enable exploration of environments too ha...

[1.2] These methods have proven effective in applications ranging from plume tracking to source seeking, w...

[1.3] The environments these robots navigate contain both scalar fields, such as temperature or chemical c...

PARAGRAPH 2 (3 citation points):
--------------------------------------------------------------------------------

[2.1] In scalar fields, every point in space is associated with a single value, such as temperature or ele...

[2.2] Navigation techniques through scalar fields are well studied, with successful demonstrations across ...

## Cell 6: Deep Paper Analysis
Analyze each kept paper in detail for citation matching

In [6]:
print("=" * 80)
print("PHASE 4: DEEP PAPER ANALYSIS")
print("=" * 80)
print(f"\nAnalyzing {len(kept_papers)} papers in detail...\n")

analysis_prompt = ChatPromptTemplate.from_messages([
    ("system", """You analyze research papers for vector field navigation literature review.
    Focus on extracting information useful for citation matching."""),
    ("human", """Analyze this paper:
    
    Filename: {filename}
    Category: {category}
    Content: {content}
    
    Return ONLY a JSON object:
    {{
        "vector_field_method": "<vector_sum | vector_to_scalar | guidance | path_following | jacobian | not_applicable | other>",
        "can_locate_critical_points": "<yes | no | not_applicable>",
        "requires_prior_knowledge": "<yes | no | unclear>",
        "requires_memory": "<yes | no | unclear>",
        "limitations": "brief description of key limitations",
        "key_contributions": "brief summary of main contributions",
        "relevant_quotes": ["quote 1", "quote 2"],
        "applications": "what domains/platforms this applies to"
    }}
    """)
])

chain = analysis_prompt | llm

paper_database = []

for i, paper in enumerate(kept_papers, 1):
    print(f"[{i}/{len(kept_papers)}] Analyzing: {paper['filename'][:60]}...")
    
    try:
        response = chain.invoke({
            "filename": paper["filename"],
            "category": paper["category"],
            "content": paper["content"]
        })
        
        analysis = extract_json(response.content)
        
        # Combine with existing paper info
        paper_entry = {
            **paper,
            **analysis
        }
        
        paper_database.append(paper_entry)
        print(f"  ✓ Method: {analysis['vector_field_method']}")
        
        time.sleep(0.3)
        
    except Exception as e:
        print(f"  ! ERROR: {e}")
        # Still add paper but with minimal analysis
        paper_database.append(paper)

# Display paper database organized by category
print("\n" + "=" * 80)
print("PAPER DATABASE")
print("=" * 80)

for cat in sorted(set(p["category"] for p in paper_database)):
    cat_papers = [p for p in paper_database if p["category"] == cat]
    print(f"\n{cat.upper().replace('_', ' ')} ({len(cat_papers)} papers):")
    print("-" * 80)
    
    for p in cat_papers:
        kitts_marker = "[KITTS]" if p.get("is_kitts") else "[NON-KITTS]"
        print(f"\n{p['filename'][:70]} {kitts_marker}")
        print(f"  Path: {p['path']}")
        if "vector_field_method" in p:
            print(f"  Method: {p['vector_field_method']}")
            print(f"  Can locate critical points: {p.get('can_locate_critical_points', 'N/A')}")
            print(f"  Contributions: {p.get('key_contributions', 'N/A')[:100]}...")

print(f"\n\n✓ Paper database built: {len(paper_database)} papers")

PHASE 4: DEEP PAPER ANALYSIS

Analyzing 37 papers in detail...

[1/37] Analyzing: [2] Citation Trakas P S Tantoulas A Bechlioulis CP Formation...
  ✓ Method: not_applicable
[2/37] Analyzing: [9] Distributed Multi-Robot Active-Sensing of a Diffusive So...
  ✓ Method: guidance
[3/37] Analyzing: [30] Multiple UAV Adaptive Navigation for Three-Dimensional ...
  ✓ Method: guidance
[4/37] Analyzing: [13] EntrapmentEscorting and Patrolling Missions - Missions....
  ✓ Method: not_applicable
[5/37] Analyzing: [52] Structured Isosurface Mapping of 3D Scalar Fields - Iso...
  ✓ Method: guidance
[6/37] Analyzing: [15] Finite-Time Estimation and Control for Multi-Aircraft S...
  ✓ Method: guidance
[7/37] Analyzing: [25] Initial Study of Multirobot Adaptive Navigation for - S...
  ✓ Method: guidance
[8/37] Analyzing: [50] Spiral Search Pattern for Scalable Assemblages of - Sea...
  ✓ Method: guidance
[9/37] Analyzing: [4] Co-Evolution of Multi-Robot Controllers and Task Cues fo...
  ✓ Method: not_ap

## Cell 7: Citation Matching with Traceability
Match each claim to relevant papers with full file path traceability

In [7]:
print("=" * 80)
print("PHASE 5: CITATION MATCHING")
print("=" * 80)

# Safety check
if len(all_claims) == 0:
    print("\n⚠️  WARNING: No claims found to match. Skipping citation matching.")
    print("   Check that Cell 5 ran successfully and found citation points.")
    citation_assignments = []
else:
    print(f"\nMatching {len(all_claims)} claims to {len(paper_database)} papers...")
    print(f"Total LLM calls needed: {len(all_claims) * len(paper_database)}")
    print("This may take 20-30 minutes...\n")

    matching_prompt = ChatPromptTemplate.from_messages([
        ("system", """You score how well a paper supports a specific claim.
        Return a relevance score (0.00-1.00) and evidence."""),
        ("human", """Does this paper support this claim?
        
        CLAIM: {claim}
        
        PAPER: {filename}
        Category: {category}
        Contributions: {contributions}
        
        Return ONLY a JSON object:
        {{
            "relevance_score": <float 0.00-1.00>,
            "evidence": "brief explanation"
        }}
        
        Scoring guide:
        - 0.90-1.00: Directly supports claim with strong evidence
        - 0.70-0.89: Good support with clear relevance
        - 0.50-0.69: Moderate support
        - 0.30-0.49: Weak/tangential support
        - 0.00-0.29: Little to no support
        """)
    ])

    chain = matching_prompt | llm

    citation_assignments = []
    citation_counter = 1

    for claim_idx, claim in enumerate(all_claims):
        print(f"\n[{claim_idx + 1}/{len(all_claims)}] Matching claim from paragraph {claim['paragraph']}...")
        print(f"  Claim: {claim['claim'][:80]}...")
        
        # Score all papers for this claim
        scores = []
        
        for paper_idx, paper in enumerate(paper_database):
            try:
                response = chain.invoke({
                    "claim": claim["claim"],
                    "filename": paper["filename"],
                    "category": paper["category"],
                    "contributions": paper.get("key_contributions", "N/A")
                })
                
                result = extract_json(response.content)
                
                scores.append({
                    "paper": paper,
                    "score": result["relevance_score"],
                    "evidence": result.get("evidence", "")
                })
                
                # Show progress every 10 papers
                if (paper_idx + 1) % 10 == 0:
                    print(f"    Progress: {paper_idx + 1}/{len(paper_database)} papers scored...")
                
                time.sleep(0.15)  # Rate limiting (reduced from 0.2)
                
            except Exception as e:
                print(f"    ! Error scoring {paper['filename'][:40]}: {e}")
                # Add a zero score so we don't skip this paper entirely
                scores.append({
                    "paper": paper,
                    "score": 0.0,
                    "evidence": f"Error during scoring: {str(e)[:50]}"
                })
                continue
        
        # Sort by score and take top N
        scores.sort(key=lambda x: x["score"], reverse=True)
        
        # Filter by threshold and take top 1-3
        top_scores = [
            s for s in scores 
            if s["score"] >= CONFIG["relevance_score_threshold"]
        ][:CONFIG["max_cites_per_claim"]]
        
        # Assign citation numbers
        claim_citations = []
        for score_entry in top_scores:
            citation_entry = {
                "citation_num": citation_counter,
                "claim": claim,
                "paper": score_entry["paper"],
                "score": score_entry["score"],
                "evidence": score_entry["evidence"]
            }
            claim_citations.append(citation_entry)
            citation_counter += 1
        
        citation_assignments.append({
            "claim": claim,
            "citations": claim_citations
        })
        
        # Fixed: Can't nest f-strings, so build the score list separately
        score_list = ', '.join([f"{c['score']:.2f}" for c in claim_citations])
        print(f"  → Assigned {len(claim_citations)} citations (scores: {score_list})")

    # Display citation assignments by paragraph
    print("\n" + "=" * 80)
    print("CITATION ASSIGNMENTS BY PARAGRAPH")
    print("=" * 80)

    for para_num in [1, 2, 3]:
        para_assignments = [a for a in citation_assignments if a["claim"]["paragraph"] == para_num]
        
        if not para_assignments:
            print(f"\nPARAGRAPH {para_num}: No citations assigned")
            continue
        
        print(f"\n{'=' * 80}")
        print(f"PARAGRAPH {para_num} CITATIONS")
        print(f"{'=' * 80}")
        
        for assignment in para_assignments:
            claim = assignment["claim"]
            citations = assignment["citations"]
            
            print(f"\nClaim: {claim['claim'][:100]}...")
            print(f"\nAssigned Citations ({len(citations)}):")
            
            for cite in citations:
                kitts_marker = "[KITTS]" if cite["paper"].get("is_kitts") else "[NON-KITTS]"
                print(f"\n  [{cite['citation_num']}] Score: {cite['score']:.2f} {kitts_marker}")
                print(f"      File: {cite['paper']['filename']}")
                print(f"      Path: {cite['paper']['path']}")
                print(f"      Evidence: {cite['evidence'][:150]}...")

    print(f"\n\n✓ Citation matching complete: {citation_counter - 1} total citations assigned")

PHASE 5: CITATION MATCHING

Matching 12 claims to 37 papers...
Total LLM calls needed: 444
This may take 20-30 minutes...


[1/12] Matching claim from paragraph 1...
  Claim: \section{Introduction}
\IEEEPARstart{M}{ultirobot} systems enable exploration of...
    Progress: 10/37 papers scored...
    Progress: 20/37 papers scored...
    Progress: 30/37 papers scored...
  → Assigned 3 citations (scores: 0.70, 0.70, 0.70)

[2/12] Matching claim from paragraph 1...
  Claim: These methods have proven effective in applications ranging from plume tracking ...
    Progress: 10/37 papers scored...
    Progress: 20/37 papers scored...
    Progress: 30/37 papers scored...
  → Assigned 3 citations (scores: 0.90, 0.85, 0.80)

[3/12] Matching claim from paragraph 1...
  Claim: The environments these robots navigate contain both scalar fields, such as tempe...
    Progress: 10/37 papers scored...
    Progress: 20/37 papers scored...
    Progress: 30/37 papers scored...
  → Assigned 3 citations (scores

## Cell 8: Generate Paragraph 3 Expansion
Create enhanced paragraph 3 based on vector field literature found

In [8]:
print("=" * 80)
print("PHASE 6: GENERATE PARAGRAPH 3 EXPANSION")
print("=" * 80)

# Get paragraph 3 citations
para3_assignments = [a for a in citation_assignments if a["claim"]["paragraph"] == 3]

# Collect all vector field papers found
vf_papers = [p for p in paper_database if p["category"] == "physical_vector_field"]

# Build context for expansion
vf_context = "\n\n".join([
    f"Paper: {p['filename']}\n"
    f"Method: {p.get('vector_field_method', 'N/A')}\n"
    f"Contributions: {p.get('key_contributions', 'N/A')}\n"
    f"Limitations: {p.get('limitations', 'N/A')}"
    for p in vf_papers
])

expansion_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are rewriting paragraph 3 of a research paper introduction.
    
    The paragraph should:
    1. Explain vector field navigation challenges
    2. Describe what methods exist in literature (vector sum, vector-to-scalar, etc.)
    3. Explain limitations of existing methods
    4. Set up the gap that the paper fills
    
    Write in IEEE Transactions formal style. Use technical language but be clear.
    Mark citation locations with [CITE: description] placeholders."""),
    ("human", """Generate an expanded paragraph 3 based on this analysis:
    
    CURRENT PARAGRAPH 3:
    {current_para3}
    
    VECTOR FIELD PAPERS FOUND:
    {vf_papers}
    
    EXISTING CLAIMS AND CITATIONS:
    {citations_info}
    
    Generate expanded paragraph 3 that:
    - Maintains the advisor-approved structure
    - Adds comprehensive coverage of vector field navigation methods found
    - Distinguishes between different approaches (vector following, vector-to-scalar, etc.)
    - Explains what each approach can and cannot do
    - Uses [CITE: paper_filename] placeholders for citations
    - Flows naturally into your contribution
    """)
])

chain = expansion_prompt | llm

# Prepare citations info
citations_info = "\n\n".join([
    f"Claim: {a['claim']['claim']}\n" +
    "Citations: " + ", ".join([f"[{c['citation_num']}] {c['paper']['filename']}" for c in a['citations']])
    for a in para3_assignments
])

try:
    response = chain.invoke({
        "current_para3": current_intro["paragraphs"][2] if len(current_intro["paragraphs"]) > 2 else "",
        "vf_papers": vf_context,
        "citations_info": citations_info
    })
    
    expanded_para3 = response.content
    
    print("\nEXPANDED PARAGRAPH 3:")
    print("=" * 80)
    print(expanded_para3)
    print("\n" + "=" * 80)
    
    # Show citation mapping for expanded paragraph
    print("\nCITATION MAPPING FOR EXPANDED PARAGRAPH 3:")
    print("=" * 80)
    for assignment in para3_assignments:
        for cite in assignment["citations"]:
            print(f"\n[{cite['citation_num']}]")
            print(f"  File: {cite['paper']['filename']}")
            print(f"  Path: {cite['paper']['path']}")
            print(f"  Kitts: {'Yes' if cite['paper'].get('is_kitts') else 'No'}")
    
except Exception as e:
    print(f"ERROR generating expansion: {e}")
    expanded_para3 = "[ERROR: Could not generate expansion]"

print("\n✓ Paragraph 3 expansion complete")

PHASE 6: GENERATE PARAGRAPH 3 EXPANSION

EXPANDED PARAGRAPH 3:
Vector field navigation presents unique challenges due to the inherent complexity of fields characterized by both magnitude and direction, particularly near critical points such as sources, sinks, saddles, and vortices [CITE: Proceedings of the ASME 2019 International Design Engineering - Testbed - 2019.pdf]. Existing methodologies in the literature can be broadly categorized into vector-following and vector-to-scalar transformation approaches. Vector-following methods, such as those employing distributed estimation strategies with Extended Information Consensus Filters [CITE: Distributed Multi-Robot Active-Sensing of a Diffusive Source - Robotics - 2025.pdf], aim to maintain the integrity of vector information. However, these methods often assume static or slowly evolving field conditions, which limits their applicability in dynamic environments where field structures change rapidly [CITE: Finite-Time Estimation and Contro

## Cell 9: Generate Complete Intro with Citations
Combine all paragraphs with citations inserted

In [9]:
print("=" * 80)
print("PHASE 7: GENERATE COMPLETE INTRODUCTION WITH CITATIONS")
print("=" * 80)

# Build complete intro
intro_with_citations = ""

# Process each paragraph
for para_num in [1, 2]:
    if para_num - 1 < len(current_intro["paragraphs"]):
        para_text = current_intro["paragraphs"][para_num - 1]
        
        # Get citations for this paragraph
        para_assignments = [a for a in citation_assignments if a["claim"]["paragraph"] == para_num]
        
        # Insert citations (simplified - mark with [N] where claims are)
        for assignment in para_assignments:
            claim_text = assignment["claim"]["claim"]
            cite_nums = [str(c["citation_num"]) for c in assignment["citations"]]
            cite_marker = f" [{','.join(cite_nums)}]"
            
            # Try to find claim in paragraph and add citation
            if claim_text in para_text:
                # Add citation at end of sentence containing claim
                para_text = para_text.replace(
                    claim_text,
                    claim_text + cite_marker,
                    1  # Replace only first occurrence
                )
        
        intro_with_citations += para_text + "\n\n"

# Add expanded paragraph 3
intro_with_citations += expanded_para3 + "\n\n"

# Add remaining paragraphs (contributions, organization)
for para_num in range(3, len(current_intro["paragraphs"])):
    intro_with_citations += current_intro["paragraphs"][para_num] + "\n\n"

# Display complete intro
print("\nCOMPLETE INTRODUCTION WITH CITATIONS:")
print("=" * 80)
print(intro_with_citations)
print("\n" + "=" * 80)

# Display complete citation mapping
print("\nCOMPLETE CITATION MAPPING (FILE TRACEABILITY):")
print("=" * 80)

# Collect all unique citations
all_citations = []
for assignment in citation_assignments:
    all_citations.extend(assignment["citations"])

# Sort by citation number
all_citations.sort(key=lambda x: x["citation_num"])

# Remove duplicates (same citation number)
seen_nums = set()
unique_citations = []
for cite in all_citations:
    if cite["citation_num"] not in seen_nums:
        unique_citations.append(cite)
        seen_nums.add(cite["citation_num"])

# Display mapping
for cite in unique_citations:
    kitts_marker = "[KITTS]" if cite["paper"].get("is_kitts") else "[NON-KITTS]"
    print(f"\n[{cite['citation_num']}] {kitts_marker}")
    print(f"  Filename: {cite['paper']['filename']}")
    print(f"  Path: {cite['paper']['path']}")
    print(f"  Used for: {cite['claim']['claim'][:80]}...")
    print(f"  Relevance: {cite['score']:.2f}")

print(f"\n\n✓ Complete introduction generated with {len(unique_citations)} unique citations")

PHASE 7: GENERATE COMPLETE INTRODUCTION WITH CITATIONS

COMPLETE INTRODUCTION WITH CITATIONS:
\section{Introduction}
\IEEEPARstart{M}{ultirobot} systems enable exploration of environments too hazardous or inaccessible for human presence, from disaster zones and chemical spills to deep ocean trenches and atmospheric phenomena [1-3]. [1,2,3] These methods have proven effective in applications ranging from plume tracking to source seeking, where the environment can be characterized by a single value at each point. [4,5,6] The environments these robots navigate contain both scalar fields, such as temperature or chemical concentration, and vector fields, such as ocean currents, wind streams, and electromagnetic fields. [7,8,9]

In scalar fields, every point in space is associated with a single value, such as temperature or elevation. [10] Navigation techniques through scalar fields are well studied, with successful demonstrations across diverse platforms including aerial drones, land rovers

## Cell 10: Vector Field Paper Catalog (ALL Papers)
Comprehensive catalog of all vector field navigation papers found (Kitts + non-Kitts)

In [10]:
print("=" * 80)
print("PHASE 8: VECTOR FIELD PAPER CATALOG")
print("=" * 80)

# Get all physical vector field papers
vf_papers = [p for p in paper_database if p["category"] == "physical_vector_field"]

# Separate by Kitts affiliation
non_kitts_vf = [p for p in vf_papers if not p.get("is_kitts")]
kitts_vf = [p for p in vf_papers if p.get("is_kitts")]

print(f"\nTotal physical vector field papers found: {len(vf_papers)}")
print(f"  Non-Kitts: {len(non_kitts_vf)}")
print(f"  Kitts: {len(kitts_vf)}")

# Display NON-KITTS papers
print("\n" + "=" * 80)
print("NON-KITTS VECTOR FIELD NAVIGATION PAPERS")
print("=" * 80)

if non_kitts_vf:
    for i, paper in enumerate(non_kitts_vf, 1):
        print(f"\n[{i}] {paper['filename']}")
        print(f"    Path: {paper['path']}")
        print(f"    Method: {paper.get('vector_field_method', 'N/A')}")
        print(f"    Can locate critical points: {paper.get('can_locate_critical_points', 'N/A')}")
        print(f"    Requires prior knowledge: {paper.get('requires_prior_knowledge', 'N/A')}")
        print(f"    Requires memory: {paper.get('requires_memory', 'N/A')}")
        print(f"    Contributions: {paper.get('key_contributions', 'N/A')}")
        print(f"    Limitations: {paper.get('limitations', 'N/A')}")
        print(f"    Applications: {paper.get('applications', 'N/A')}")
        print(f"    Relevance to your work: {paper['reason']}")
else:
    print("\n⚠️  NO NON-KITTS VECTOR FIELD PAPERS FOUND")
    print("    This may indicate:")
    print("    - Vector field navigation for critical point detection is truly novel")
    print("    - Papers in collection focus on other aspects")
    print("    - Pre-filtering was too aggressive")

# Display KITTS papers
print("\n" + "=" * 80)
print("KITTS-AFFILIATED VECTOR FIELD PAPERS")
print("=" * 80)

if kitts_vf:
    for i, paper in enumerate(kitts_vf, 1):
        print(f"\n[{i}] {paper['filename']}")
        print(f"    Path: {paper['path']}")
        print(f"    Method: {paper.get('vector_field_method', 'N/A')}")
        print(f"    Contributions: {paper.get('key_contributions', 'N/A')}")
        print(f"    Relevance: {paper['reason']}")
else:
    print("\n(No Kitts-affiliated vector field papers found in collection)")

# Summary insights
print("\n" + "=" * 80)
print("LITERATURE LANDSCAPE INSIGHTS")
print("=" * 80)

print("\nVector Field Methods Found:")
methods = {}
for p in vf_papers:
    method = p.get('vector_field_method', 'unknown')
    if method not in methods:
        methods[method] = 0
    methods[method] += 1

for method, count in methods.items():
    print(f"  - {method}: {count} papers")

print("\nCritical Point Detection Capability:")
can_detect = sum(1 for p in vf_papers if p.get('can_locate_critical_points') == 'yes')
cannot_detect = sum(1 for p in vf_papers if p.get('can_locate_critical_points') == 'no')
print(f"  - Can locate critical points: {can_detect} papers")
print(f"  - Cannot locate critical points: {cannot_detect} papers")
print(f"  - Your contribution fills gap: {'YES' if cannot_detect > 0 else 'Unclear'}")

print("\n✓ Vector field paper catalog complete")

PHASE 8: VECTOR FIELD PAPER CATALOG

Total physical vector field papers found: 7
  Non-Kitts: 3
  Kitts: 4

NON-KITTS VECTOR FIELD NAVIGATION PAPERS

[1] [9] Distributed Multi-Robot Active-Sensing of a Diffusive Source - Robotics - 2025.pdf
    Path: ./Papers/[9] Distributed Multi-Robot Active-Sensing of a Diffusive Source - Robotics - 2025.pdf
    Method: guidance
    Can locate critical points: yes
    Requires prior knowledge: no
    Requires memory: yes
    Contributions: The paper introduces a distributed estimation strategy using an Extended Information Consensus Filter with a forgetting factor, and a decentralized motion strategy to minimize a Gramian-based information metric. It integrates collision avoidance through Control Barrier Functions in a Quadratic Program.
    Limitations: The method assumes a continuously releasing source, which may not be applicable to scenarios with instantaneous releases. The field gradient can become unbounded, posing challenges in certain enviro

Vector fields, prevalent in maritime and atmospheric applications, present fundamentally different navigation challenges. Unlike scalar fields, vector fields possess both magnitude and direction at each point, creating complex dynamics near critical points where sources, sinks, saddles, and vortices exhibit distinct local behaviors [CITE: Proceedings of the ASME 2019 International Design Engineering - Testbed - 2019.pdf]. Methods that preserve this vector information typically require extensive measurement histories, detailed prior field models, or dense spatial sampling [CITE: Distributed Multi-Robot Active-Sensing of a Diffusive Source - Robotics - 2025.pdf], [CITE: Finite-Time Estimation and Control for Multi-Aircraft Systems - Estimation - 20.pdf]. These requirements become problematic in environments where prior field knowledge is unavailable. To circumvent these constraints, some approaches extract scalar quantities such as magnitude or vorticity and apply established gradient-based navigation primitives [CITE: Gradient-Based Cluster Space Navigation for Autonomous Surface Vessels - Transactions - 2013.pdf]. While effective for coarse directional guidance, these dimensionality-reduction methods cannot recover the precise location of critical points from local measurements alone [CITE: Initial Study of Multirobot Adaptive Navigation for - Study.pdf].